# 01 — Data shape and sanity checks

DVF+ geocoded property sales, 2021–2025, loaded as partitioned Parquet.

In [ ]:
import duckdb, pandas as pd
from pathlib import Path
PARQUET_GLOB = str(Path.cwd().parent / "data" / "parquet" / "year=*" / "*.parquet")
con = duckdb.connect()
con.execute(f"CREATE VIEW dvf AS SELECT * FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=true)")
print("Row count:", con.execute("SELECT COUNT(*) FROM dvf").fetchone()[0])


## 1. Rows, unique mutations, multi-row ratio
DVF+ repeats sale data across rows when a sale includes multiple lots/parcels. `id_mutation` is the authoritative sale key.

In [ ]:
con.execute('''
SELECT year, COUNT(*) AS rows, COUNT(DISTINCT id_mutation) AS mutations,
       ROUND(COUNT(*)::DOUBLE / COUNT(DISTINCT id_mutation), 3) AS rows_per_mutation
FROM dvf GROUP BY year ORDER BY year
''').df()

## 2. How lumpy are sales? Distribution of rows-per-mutation

In [ ]:
con.execute('''
WITH per_mut AS (SELECT id_mutation, COUNT(*) AS n FROM dvf GROUP BY id_mutation)
SELECT n AS rows_per_mutation, COUNT(*) AS n_mutations
FROM per_mut GROUP BY n ORDER BY n LIMIT 20
''').df()

## 2b. Extreme outliers: biggest multi-row mutations
These are portfolio / bulk real-estate deals — NOT typical residential sales. Probably worth filtering out for most analyses.

In [ ]:
con.execute('''
SELECT id_mutation, COUNT(*) AS n_rows,
       MIN(valeur_fonciere) AS valeur,
       COUNT(DISTINCT code_commune) AS n_communes
FROM dvf GROUP BY id_mutation ORDER BY n_rows DESC LIMIT 10
''').df()

## 3. Consistency within a mutation
Does valeur / date / commune stay stable across rows of the same id_mutation?

In [ ]:
con.execute('''
SELECT
  (SELECT COUNT(*) FROM (SELECT id_mutation FROM dvf WHERE valeur_fonciere IS NOT NULL
                         GROUP BY id_mutation HAVING COUNT(DISTINCT valeur_fonciere) > 1))
    AS inconsistent_valeur,
  (SELECT COUNT(*) FROM (SELECT id_mutation FROM dvf
                         GROUP BY id_mutation HAVING COUNT(DISTINCT date_mutation) > 1))
    AS inconsistent_date,
  (SELECT COUNT(*) FROM (SELECT id_mutation FROM dvf
                         GROUP BY id_mutation HAVING COUNT(DISTINCT code_commune) > 1))
    AS multi_commune
''').df()

## 4. Nature mutation distribution

In [ ]:
con.execute('''
SELECT nature_mutation, COUNT(DISTINCT id_mutation) AS mutations
FROM dvf GROUP BY nature_mutation ORDER BY mutations DESC
''').df()

## 5. Type local distribution (row-level)
NULL type_local = terrain/parcel-only rows (no building). These account for ~40% of rows.

In [ ]:
con.execute('''
SELECT COALESCE(type_local, '(null — terrain/parcel)') AS type_local, COUNT(*) AS rows
FROM dvf GROUP BY type_local ORDER BY rows DESC
''').df()

## 6. Missingness on key columns

In [ ]:
con.execute('''
SELECT COUNT(*) AS total_rows,
  SUM(valeur_fonciere IS NULL)::BIGINT AS null_valeur,
  SUM(latitude IS NULL)::BIGINT AS null_lat,
  SUM(type_local IS NULL)::BIGINT AS null_type_local,
  SUM(surface_reelle_bati IS NULL)::BIGINT AS null_bati
FROM dvf
''').df()

## 7. Valeur fonciere distribution (sale-level)

In [ ]:
con.execute('''
WITH sales AS (
  SELECT id_mutation, ANY_VALUE(valeur_fonciere) AS valeur
  FROM dvf WHERE valeur_fonciere IS NOT NULL GROUP BY id_mutation)
SELECT COUNT(*) AS sales, MIN(valeur) AS min,
       ROUND(QUANTILE_CONT(valeur, 0.01)) AS p01,
       ROUND(QUANTILE_CONT(valeur, 0.50)) AS p50,
       ROUND(QUANTILE_CONT(valeur, 0.99)) AS p99,
       ROUND(QUANTILE_CONT(valeur, 0.999)) AS p999,
       MAX(valeur) AS max
FROM sales
''').df()

Biggest sales — sniff test for data errors:

In [ ]:
con.execute('''
SELECT id_mutation, ANY_VALUE(valeur_fonciere) AS valeur,
       ANY_VALUE(nom_commune) AS commune, ANY_VALUE(date_mutation) AS date
FROM dvf GROUP BY id_mutation ORDER BY valeur DESC NULLS LAST LIMIT 10
''').df()

## 8. Geocoding completeness
DVF+ is pre-geocoded but not all rows have lat/lon. ~2% missing. `lat_out_of_france` rows are DOM-TOM (French overseas territories).

In [ ]:
con.execute('''
SELECT SUM(latitude IS NOT NULL)::BIGINT AS geocoded,
       SUM(latitude IS NULL)::BIGINT AS missing,
       SUM(latitude NOT BETWEEN 41 AND 52)::BIGINT AS lat_out_of_france
FROM dvf
''').df()

## 9. Rooms vs bedrooms?
`nombre_pieces_principales` peaks at 4 for Maisons (T4 is very common) — consistent with *rooms* (habitable rooms incl. living room), not bedrooms. `pieces >= 5` ≈ *at least 4 bedrooms*.

In [ ]:
import matplotlib.pyplot as plt
df = con.execute('''
SELECT nombre_pieces_principales AS pieces, COUNT(*) AS rows
FROM dvf WHERE type_local='Maison' AND nombre_pieces_principales BETWEEN 1 AND 12
GROUP BY pieces ORDER BY pieces
''').df()
df.plot.bar(x='pieces', y='rows', legend=False, figsize=(8,3), title='Maison rooms distribution')
plt.show()